# Sentinel-2 ingest and ICESat-2 fusion on a shared HEALPix grid

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/sentinel2_fusion.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/sentinel2_fusion.ipynb):
every byte it reads is anonymous -- Sentinel-2 metadata from the
[Earth Search](https://element84.com/earth-search/) STAC API, imagery from the
public `e84-earth-search-sentinel-data` COG bucket, and ICESat-2 aggregates
from a public [source.coop](https://source.coop/englacial/zagg/benchmarks)
store. No credentials, no AWS account._

This example is the [issue #218](https://github.com/englacial/zagg/issues/218)
template in action. It ingests Sentinel-2 L2A surface reflectance onto the
**same HEALPix order-19 cells (~12.4 m)** that zagg's ICESat-2 products
aggregate to, using **pull-NN sampling**: every cell takes the source pixel
nearest its center. That one design choice buys three things at once:

* **dense by construction** -- every covered cell gets a value at *any* cell
  order (push-hashing pixel centers instead leaves 2--4% holes, because
  HEALPix cells are equal-area but not equal-shape);
* **exact DNs** -- the most-central measurement is selected, never averaged,
  so uint16 digital numbers survive bit-for-bit and categorical bands (the
  scene classification, `scl`) stay categorical;
* **trivial fusion** -- Sentinel-2 and ICESat-2 land in the same cell-ID
  space, so coincident extraction is an integer set intersection. No
  reprojection, no interpolation, no geometry at read time.

The output layout is one **`(time, cells)`** array per band -- time chunks of
1, so each timestep x chunk is one storage object -- with a real `time`
coordinate. Timesteps are Sentinel-2 *datatakes* (`s2:datatake_id`), not item
datetimes: adjacent MGRS tiles of one datatake are a single timestep, and
cells in the 9.8 km tile overlap take the value from the **nearest tile
center**.


## 1. The fusion target: ATL06 cycle 22, published on source.coop

The join target is a public, anonymous ATL06 (land-ice height) aggregation
for ICESat-2 **cycle 22** (Jan--Apr 2024) at HEALPix **order 12** (~1.6 km),
nested ids, fullsphere layout. Our Sentinel-2 ingest below lands at order
**19** (~12.4 m) in the same nested-id hierarchy, so the cross-resolution
join is *integer arithmetic*: an order-19 cell's order-12 parent is
``nested_19 // 4**7`` -- one shift, no geometry.

We work over the Larsen B embayment on the Antarctic Peninsula, in the same
austral summer the ICESat-2 cycle spans.


In [ ]:
import numpy as np

# Larsen B embayment, Antarctic Peninsula (lon_min, lat_min, lon_max, lat_max).
BBOX = (-62.45, -65.55, -62.15, -65.35)
ATL06_STORE = "s3://us-west-2.opendata.source.coop/englacial/zagg/benchmarks/atl06_cycle22_fullsphere.zarr"


## 2. Catalog: query Earth Search

`STACSource` speaks to any STAC API root. For Sentinel-2 we query the
Collection-1 reprocessing (`sentinel-2-c1-l2a`) together with
`sentinel-2-pre-c1-l2a` (identical schema; fills the archive where the
reprocessing hasn't reached yet), keep only the assets we'll ingest, and set
`time_key` so acquisition grouping happens by datatake.


In [ ]:
from zagg.catalog.sources import STACQuery, STACSource

src = STACSource(
    "https://earth-search.aws.element84.com/v1",
    assets=["red", "nir", "scl"],
    time_key="s2:datatake_id",
)
query = STACQuery(
    collections=["sentinel-2-c1-l2a", "sentinel-2-pre-c1-l2a"],
    start_date="2023-11-01",
    end_date="2024-03-31",
    region=BBOX,
    max_cloud_cover=40,
)
cat = src.fetch(query)
for rec in cat.granule_records()[:8]:
    print(rec["id"], rec["datetime"][:19], "->", rec["time_key"])


## 3. Shardmap and ingest

The shipped `sentinel2_l2a` config declares the whole pipeline: bands with
dtypes and CF `scale_factor`/`add_offset` attrs (reflectance = DN x 1e-4 -
0.1 -- recorded, never applied), nodata, and the parent-11/child-19 HEALPix
grid. `ShardMap.build` intersects the MGRS tile footprints with the grid
exactly as it does ICESat-2 swaths; `agg` then runs one worker per shard:
open each band's COG header, fetch only the ~3 km window of 1024-px DEFLATE
tiles the shard touches (async-tiff -- no GDAL anywhere), gather the nearest
pixel per cell, and write `(time, cells)` slabs.

To keep Binder light we ingest the first two datatakes over the AOI.


In [ ]:
from zagg.catalog.shardmap import ShardMap
from zagg.config import default_config, validate_config
from zagg.grids import from_config

config = default_config("sentinel2_l2a")
config.data_source["bands"] = {
    k: v for k, v in config.data_source["bands"].items() if k in ("red", "nir", "scl")
}
config.output["store"] = "sentinel2_larsenb.zarr"
validate_config(config)

grid = from_config(config)
sm = ShardMap.build(cat, grid)  # coverage from the catalog bbox

# Keep the first two datatakes (chronological) so the Binder run stays small.
keep = sorted({e["time_key"] for g in sm.granules for e in g})[:2]
sm.granules = [[e for e in g if e["time_key"] in keep] for g in sm.granules]
sm.to_json("shardmap_s2_larsenb.json")
print(f"{len(sm.shard_keys)} shards, datatakes: {keep}")


In [ ]:
from zagg.runner import agg

summary = agg(
    config,
    catalog="shardmap_s2_larsenb.json",
    backend="local",
    max_workers=4,
    overwrite=True,
)
summary


## 4. Read the `(time, cells)` product

The store is plain Zarr v3: one 2-D array per band plus `time` (microseconds
since epoch) and `cell_ids` (nested order-19 ids). The arrays logically span
the full sphere; we slice the shard ranges we wrote. Reflectance decode is
the CF attrs applied by hand -- the stored values are exact source DNs.


In [ ]:
import xarray as xr

from zagg import open_store

s2 = xr.open_dataset(
    open_store(config.output["store"], read_only=True),
    engine="zarr",
    consolidated=False,
    zarr_format=3,
    group=grid.group_path,
)

shard_ranges = [
    (int(grid.block_index(int(s))[0]) * grid.cells_per_shard, int(s)) for s in sm.shard_keys
]


def gather(name, t=None):
    """Concatenate the written shard ranges of one array (one timestep)."""
    parts = []
    for start, _s in shard_ranges:
        sl = slice(start, start + grid.cells_per_shard)
        parts.append((s2[name][t, sl] if t is not None else s2[name][sl]).values)
    return np.concatenate(parts)


times = s2["time"].values.astype("datetime64[us]")
s2_ids = gather("cell_ids")
red0 = gather("red", 0)
nir0 = gather("nir", 0)
scl0 = gather("scl", 0)
print("timesteps:", list(times))
print(f"cells written: {s2_ids.size:,}; valid at t0: {(red0 > 0).sum():,}")


In [ ]:
import matplotlib.pyplot as plt
from mortie import mort2geo

cells_morton = np.concatenate([grid.children(s) for _start, s in shard_ranges])
clats, clons = mort2geo(cells_morton)
ok = red0 > 0  # DN 0 is fill/nodata
refl = red0[ok].astype(float) * 1e-4 - 0.1

fig, ax = plt.subplots(figsize=(8, 6))
pc = ax.scatter(clons[ok], clats[ok], c=refl, s=1.5, cmap="RdYlGn_r", vmin=0, vmax=0.35)
fig.colorbar(pc, label="red surface reflectance")
ax.set_title(f"Sentinel-2 red on HEALPix order 19 -- {times[0]}")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
plt.show()


## 5. Fusion: the cross-resolution join is a bit shift

Both stores carry **nested HEALPix ids**. The order-19 Sentinel-2 cells roll
up to their order-12 ATL06 parents by integer division (``4**(19-12)``
children per parent), and the fullsphere layout means an order-12 array
index *is* the nested id -- so "read the ICESat-2 cells under my imagery"
is a fancy-indexed Zarr read, nothing more.


In [ ]:
from mortie import clip2order, mort2healpix

# Order-12 parents of every ingested S2 cell (morton -> nested id).
parents_morton = np.unique(clip2order(12, cells_morton))
parents_nested, _order = mort2healpix(parents_morton)
parents_nested = np.sort(parents_nested.astype(np.int64))

atl06 = xr.open_dataset(
    open_store(ATL06_STORE, read_only=True, skip_signature=True),
    engine="zarr",
    consolidated=False,
    zarr_format=3,
    group="12",
)
h_mean = atl06["h_mean"][parents_nested].values
is2_count = atl06["count"][parents_nested].values
measured = is2_count > 0
print(f"order-12 parents under the imagery: {parents_nested.size:,}")
print(f"...with ICESat-2 cycle-22 heights:  {measured.sum():,}")


In [ ]:
# Roll the imagery up to the same parents (analysis-time mean of the exact
# per-cell DNs) and compare against the coincident ATL06 heights.
parent_of_cell, _ = mort2healpix(clip2order(12, cells_morton))
pos = np.searchsorted(parents_nested, parent_of_cell.astype(np.int64))

valid = red0 > 0
refl_all = red0.astype(float) * 1e-4 - 0.1
sums = np.bincount(pos[valid], weights=refl_all[valid], minlength=parents_nested.size)
ns = np.bincount(pos[valid], minlength=parents_nested.size)
mean_refl = np.divide(sums, ns, out=np.full_like(sums, np.nan), where=ns > 0)

both = measured & (ns > 0)
print(f"parents with BOTH imagery and altimetry: {both.sum():,}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mean_refl[both], h_mean[both], s=8, alpha=0.6)
ax.set_xlabel(f"Sentinel-2 red reflectance (order-12 mean) -- {times[0]}")
ax.set_ylabel("ATL06 h_mean (m), cycle 22")
ax.set_title("Coincident cells, joined by integer id -- zero resampling")
plt.show()


## 6. Where this goes

* **Training loaders**: the band arrays are chunked `(1, 65536)` -- one
  storage object per (timestep, shard). A PyTorch `DataLoader` can range-read
  chunk `k` from every aligned array (`red`, `nir`, `scl`, the ICESat-2
  aggregates) and get co-registered tensors with no geospatial code in the
  loop: `np.asarray` of a Zarr block, straight to `torch.from_numpy`.
  The scale/offset decode is one fused multiply-add on the GPU.
* **More timesteps**: appends *will* land as the standard Zarr
  resize-then-write-slab pattern (a runner-owned resize; chunk keys are
  `t/chunk`, so nothing already written is rewritten) -- tracked on
  [issue #218](https://github.com/englacial/zagg/issues/218). Today a
  changed time range is a full re-run with `overwrite=True`.
* **Same-order joins**: against an order-19 product (e.g. the ATL03
  photon t-digests), the join needs no roll-up at all -- it is
  `np.intersect1d` on `cell_ids`.
* **Full-scale runs**: the same config fans out one shard per AWS Lambda --
  see the [deployment docs](../docs/deployment/lambda.md). The order-19 vs
  order-20 storage trade and the pull-NN design measurements live on
  [issue #218](https://github.com/englacial/zagg/issues/218).
